# SurfaceTarget-CRC — phase 02 — patch 2 2

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 2 — PATCH 2.2 (CORRECTED)
#
# CPTAC MSS PROTEIN ENRICHMENT AND IDENTIFIER AUDIT
#
# Objectives:
# 1. Validate previously downloaded CPTAC source files
# 2. Audit tumor/normal/paired sample identifiers
# 3. Correct clinical patient identifier handling
# 4. Select published MSS paired-proteomics patients
# 5. Analyze reported tumor-normal protein log2 ratios
# 6. Apply multiple-testing correction
# 7. Integrate transcriptomic screening evidence
# 8. Preserve missing-measurement information
# 9. Generate an exploratory proteomics figure
# 10. Save results, provenance, and research checkpoint
#
# CORRECTIONS:
# - Handle clinical patient ID saved as "index",
#   "Unnamed: 0", or "cptac_patient_id".
# - Explicitly validate clinical/sample correspondence.
# - Use strictly Boolean NumPy masks in np.select.
# - Preserve missing proteomics evidence as missing.
#
# IMPORTANT:
# - No protein imputation
# - No TCGA/CPTAC patient-level merging
# - No separate tumor-normal matrix subtraction
# - No therapeutic target validation
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND PROJECT CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, binomtest
from statsmodels.stats.multitest import multipletests

import hashlib
import json
import re


ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/cptac_colon"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    META,
    RAW,
    RESULTS,
    FIGURES,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


SOURCE_MANIFEST = (
    META / "cptac_colon_source_manifest.csv"
)

PAIRED_METADATA = (
    META / "cptac_colon_paired_sample_metadata.csv"
)

TRANSCRIPTOMIC_PATH = (
    RESULTS / "tcga_coad_paired_support_comparison.csv"
)

MIN_OBSERVED_MSS = 30


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 2 — PATCH 2.2")
print("CPTAC MSS PROTEIN ENRICHMENT")
print("=" * 65)


# ------------------------------------------------------------
# 2. SOURCE DATA INTEGRITY VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SOURCE DATA INTEGRITY VALIDATION")
print("-" * 65)


if not SOURCE_MANIFEST.exists():
    raise FileNotFoundError(
        "Patch 2.1 source manifest is missing."
    )


source_manifest = pd.read_csv(
    SOURCE_MANIFEST,
    dtype=str,
    keep_default_na=False
)


expected_datasets = {
    "tumor_proteomics",
    "normal_proteomics",
    "paired_proteomics",
    "clinical"
}


if set(source_manifest["dataset"]) != expected_datasets:
    raise RuntimeError(
        "Unexpected CPTAC source manifest."
    )


if source_manifest["dataset"].duplicated().any():
    raise RuntimeError(
        "Duplicate datasets in source manifest."
    )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source_paths = {}

for _, row in source_manifest.iterrows():

    path = RAW / row["filename"]

    if not path.exists():
        raise FileNotFoundError(path)

    actual_sha = sha256_file(path)

    if actual_sha.lower() != row["sha256"].lower():

        raise RuntimeError(
            "Source checksum mismatch: "
            f"{row['dataset']}"
        )

    source_paths[row["dataset"]] = path


print("\nAll four source files verified.")

print(
    "Checksums match the source manifest "
    "recorded in Patch 2.1."
)


# ------------------------------------------------------------
# 3. LOAD PROTEOMICS MATRICES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CPTAC PROTEOMICS MATRICES")
print("-" * 65)


def load_matrix(dataset):

    df = pd.read_csv(
        source_paths[dataset],
        sep="\t",
        index_col=0,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    df.index = (
        df.index.astype(str).str.strip()
    )

    df.columns = (
        df.columns.astype(str).str.strip()
    )

    if df.index.duplicated().any():

        raise RuntimeError(
            f"Duplicate protein identifiers: {dataset}"
        )

    if df.columns.duplicated().any():

        raise RuntimeError(
            f"Duplicate sample identifiers: {dataset}"
        )

    if len(df) == 0 or len(df.columns) == 0:

        raise RuntimeError(
            f"Empty proteomics matrix: {dataset}"
        )

    numeric = df.apply(
        pd.to_numeric,
        errors="coerce"
    )

    invalid = (
        df.notna() & numeric.isna()
    )

    if invalid.to_numpy().any():

        raise RuntimeError(
            f"Unexpected nonnumeric measurements: {dataset}"
        )

    values = numeric.to_numpy(dtype=float)

    if np.isinf(values).any():

        raise RuntimeError(
            f"Infinite measurements detected: {dataset}"
        )

    return numeric


tumor = load_matrix("tumor_proteomics")

normal = load_matrix("normal_proteomics")

paired = load_matrix("paired_proteomics")


print("\nProteomics matrices loaded.")

print("Tumor:", tumor.shape)

print("Normal:", normal.shape)

print("Paired ratios:", paired.shape)


if tumor.shape != (8067, 97):

    raise RuntimeError(
        "Unexpected tumor proteomics dimensions."
    )

if normal.shape != (8067, 100):

    raise RuntimeError(
        "Unexpected normal proteomics dimensions."
    )

if paired.shape != (6422, 96):

    raise RuntimeError(
        "Unexpected paired proteomics dimensions."
    )


print("\nProteomics matrix validation: PASSED")


# ------------------------------------------------------------
# 4. CPTAC SAMPLE IDENTIFIER AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CPTAC SAMPLE IDENTIFIER AUDIT")
print("-" * 65)


print("\nFirst 8 tumor identifiers:")
print(tumor.columns[:8].tolist())

print("\nFirst 8 normal identifiers:")
print(normal.columns[:8].tolist())

print("\nFirst 8 paired-ratio identifiers:")
print(paired.columns[:8].tolist())


# CPTAC colon sample IDs may contain:
#
# 01CO005  = tumor/patient identifier
# 01CO005N = corresponding normal specimen
#
# Extract the shared patient identifier.
#
# This establishes identifier correspondence,
# not sample-processing equivalence.


PATIENT_PATTERN = re.compile(
    r"\d{2}CO\d{3}"
)


def extract_patient_id(sample_id):

    match = PATIENT_PATTERN.search(
        str(sample_id)
    )

    if match is None:
        return None

    return match.group(0)


identifier_records = []

for dataset_name, matrix in [
    ("Tumor", tumor),
    ("Normal", normal),
    ("Paired", paired)
]:

    for sample_id in matrix.columns:

        identifier_records.append({

            "dataset": dataset_name,

            "original_sample_id": sample_id,

            "extracted_patient_id":
                extract_patient_id(sample_id)

        })


identifier_audit = pd.DataFrame(
    identifier_records
)


identifier_summary = (
    identifier_audit.groupby("dataset")
    .agg(

        samples=(
            "original_sample_id",
            "count"
        ),

        mapped_patients=(
            "extracted_patient_id",
            "nunique"
        ),

        unmatched=(
            "extracted_patient_id",
            lambda x: int(x.isna().sum())
        )

    )
)


print("\nPatient identifier extraction:")

print(identifier_summary.to_string())


complete_mapping = (
    identifier_audit["extracted_patient_id"]
    .notna()
    .all()
)


unique_mapping = all(

    not group["extracted_patient_id"]
    .duplicated()
    .any()

    for _, group in identifier_audit.groupby(
        "dataset"
    )

)


if complete_mapping and unique_mapping:

    patient_sets = {

        name: set(group["extracted_patient_id"])

        for name, group in identifier_audit.groupby(
            "dataset"
        )

    }

    tumor_normal_overlap = len(
        patient_sets["Tumor"]
        & patient_sets["Normal"]
    )

    paired_tumor_overlap = len(
        patient_sets["Paired"]
        & patient_sets["Tumor"]
    )

    paired_normal_overlap = len(
        patient_sets["Paired"]
        & patient_sets["Normal"]
    )

    all_three_overlap = len(
        patient_sets["Tumor"]
        & patient_sets["Normal"]
        & patient_sets["Paired"]
    )

    print("\nPatient-level overlaps:")

    print(
        "Tumor and normal:",
        tumor_normal_overlap
    )

    print(
        "Paired and tumor:",
        paired_tumor_overlap
    )

    print(
        "Paired and normal:",
        paired_normal_overlap
    )

    print(
        "All three:",
        all_three_overlap
    )

else:

    print(
        "\nWARNING: One-to-one patient "
        "identifier mapping could not "
        "be established."
    )


print("\nNo separate tumor-normal subtraction performed.")


# ============================================================
# 5. CPTAC MSS CLINICAL COHORT IDENTIFICATION
#
# CORRECTED SECTION
#
# The CSV saved by Patch 2.1 may contain:
#
# - cptac_patient_id
# - index
# - Unnamed: 0
#
# This section detects and standardizes
# the identifier column before continuing.
# ============================================================

print("\n" + "-" * 65)
print("MSS PROTEOMICS COHORT")
print("-" * 65)


if not PAIRED_METADATA.exists():

    raise FileNotFoundError(
        "CPTAC paired clinical metadata missing."
    )


clinical = pd.read_csv(
    PAIRED_METADATA,
    dtype=str,
    keep_default_na=False
)


print("\nSaved clinical metadata columns:")

print(clinical.columns.tolist())


# ------------------------------------------------------------
# 5A. Identify patient ID column
# ------------------------------------------------------------

possible_id_columns = [
    "cptac_patient_id",
    "index",
    "Unnamed: 0"
]


available_id_columns = [
    column
    for column in possible_id_columns
    if column in clinical.columns
]


if len(available_id_columns) != 1:

    raise RuntimeError(
        "Could not uniquely identify the "
        "CPTAC patient identifier column. "
        f"Candidates found: {available_id_columns}"
    )


original_id_column = available_id_columns[0]


clinical = clinical.rename(
    columns={
        original_id_column: "cptac_patient_id"
    }
)


print(
    "\nOriginal patient ID column:",
    original_id_column
)

print(
    "Standardized patient ID column:",
    "cptac_patient_id"
)


# ------------------------------------------------------------
# 5B. Validate clinical fields
# ------------------------------------------------------------

required_clinical = {
    "cptac_patient_id",
    "msi_classification"
}


missing_columns = (
    required_clinical - set(clinical.columns)
)


if missing_columns:

    raise RuntimeError(
        "Missing required clinical columns: "
        f"{missing_columns}"
    )


clinical["cptac_patient_id"] = (
    clinical["cptac_patient_id"]
    .astype(str)
    .str.strip()
)


clinical["msi_classification"] = (
    clinical["msi_classification"]
    .astype(str)
    .str.strip()
    .str.upper()
)


if clinical["cptac_patient_id"].eq("").any():

    raise RuntimeError(
        "Empty CPTAC patient identifiers detected."
    )


if clinical["cptac_patient_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate CPTAC clinical patient identifiers."
    )


clinical = clinical.set_index(
    "cptac_patient_id"
)


# ------------------------------------------------------------
# 5C. Validate paired-proteomics correspondence
# ------------------------------------------------------------

missing_patients = (
    set(paired.columns)
    - set(clinical.index)
)


if missing_patients:

    raise RuntimeError(
        "Paired-ratio patients missing clinical "
        "metadata: "
        f"{sorted(missing_patients)}"
    )


print("\nClinical patient correspondence: PASSED")


# ------------------------------------------------------------
# 5D. Identify published MSS patients
# ------------------------------------------------------------

msi_status = (
    clinical.loc[
        paired.columns,
        "msi_classification"
    ]
)


print("\nMSI classifications:")

print(
    msi_status.value_counts().to_string()
)


mss_patients = msi_status.index[
    msi_status.eq("MSS")
].tolist()


if len(mss_patients) != 73:

    raise RuntimeError(
        "Expected 73 published MSS patients. "
        f"Found {len(mss_patients)}."
    )


mss = paired.loc[
    :,
    mss_patients
].copy()


print("\nMSS patients:", len(mss_patients))

print("Measured protein rows:", len(mss))

print("\nMSS cohort identification: PASSED")


# ------------------------------------------------------------
# 6. CALCULATE MSS PROTEIN ENRICHMENT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING MSS PROTEIN ENRICHMENT")
print("-" * 65)


# The paired matrix contains reported
# tumor-normal protein log2 ratios.
#
# Positive ratio = higher reported abundance
# in tumor relative to normal.
#
# Missing measurements are excluded
# individually for each protein.
#
# No missing values are replaced with zero.


protein_records = []


for gene_symbol, row in mss.iterrows():

    values = row.to_numpy(
        dtype=float
    )

    observed = values[
        np.isfinite(values)
    ]

    n = len(observed)

    # --------------------------------------------
    # Descriptive protein statistics
    # --------------------------------------------

    if n > 0:

        median_ratio = float(
            np.median(observed)
        )

        mean_ratio = float(
            np.mean(observed)
        )

        fraction_positive = float(
            np.mean(observed > 0)
        )

        fraction_ge1 = float(
            np.mean(observed >= 1)
        )

        positive = int(
            np.count_nonzero(observed > 0)
        )

        negative = int(
            np.count_nonzero(observed < 0)
        )

        zero = int(
            np.count_nonzero(observed == 0)
        )

    else:

        median_ratio = np.nan

        mean_ratio = np.nan

        fraction_positive = np.nan

        fraction_ge1 = np.nan

        positive = 0
        negative = 0
        zero = 0

    # --------------------------------------------
    # Statistical tests
    # --------------------------------------------

    wilcoxon_p = np.nan

    sign_p = np.nan


    if n >= MIN_OBSERVED_MSS:

        if np.all(observed == 0):

            wilcoxon_p = 1.0

        else:

            wilcoxon_p = float(
                wilcoxon(
                    observed,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                ).pvalue
            )

        nonzero = positive + negative

        if nonzero > 0:

            sign_p = float(
                binomtest(
                    positive,
                    n=nonzero,
                    p=0.5,
                    alternative="two-sided"
                ).pvalue
            )

        else:

            sign_p = 1.0


    protein_records.append({

        "protein_gene_symbol": gene_symbol,

        "mss_patients_total": len(mss_patients),

        "mss_measurements": n,

        "missing_mss_measurements":
            len(mss_patients) - n,

        "mss_measurement_fraction":
            n / len(mss_patients),

        "median_tumor_normal_log2ratio":
            median_ratio,

        "mean_tumor_normal_log2ratio":
            mean_ratio,

        "fraction_tumor_higher":
            fraction_positive,

        "fraction_log2ratio_ge1":
            fraction_ge1,

        "positive_patients":
            positive,

        "negative_patients":
            negative,

        "zero_ratio_patients":
            zero,

        "wilcoxon_pvalue":
            wilcoxon_p,

        "sign_test_pvalue":
            sign_p

    })


protein_stats = pd.DataFrame(
    protein_records
)


print("\nProtein statistics calculated.")

print("Total protein rows:", len(protein_stats))

print(
    "Minimum required MSS measurements:",
    MIN_OBSERVED_MSS
)


# ------------------------------------------------------------
# 7. MULTIPLE-TESTING CORRECTION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MULTIPLE-TESTING CORRECTION")
print("-" * 65)


protein_stats["wilcoxon_fdr"] = np.nan

protein_stats["sign_test_fdr"] = np.nan


for p_column, fdr_column in [

    ("wilcoxon_pvalue", "wilcoxon_fdr"),

    ("sign_test_pvalue", "sign_test_fdr")

]:

    valid = (
        protein_stats[p_column]
        .notna()
        .to_numpy(dtype=bool)
    )

    if valid.sum() == 0:

        raise RuntimeError(
            f"No valid protein tests: {p_column}"
        )

    adjusted = multipletests(

        protein_stats.loc[
            valid,
            p_column
        ].to_numpy(dtype=float),

        method="fdr_bh"

    )[1]


    protein_stats.loc[
        valid,
        fdr_column
    ] = adjusted


# ------------------------------------------------------------
# 7A. Apply exploratory enrichment criteria
# ------------------------------------------------------------

protein_stats["protein_enriched"] = (

    protein_stats["wilcoxon_fdr"].lt(0.05)

    & protein_stats[
        "median_tumor_normal_log2ratio"
    ].ge(1.0)

    & protein_stats[
        "fraction_tumor_higher"
    ].ge(0.70)

    & protein_stats[
        "mss_measurements"
    ].ge(MIN_OBSERVED_MSS)

)


proteins_tested = int(
    protein_stats["wilcoxon_pvalue"]
    .notna()
    .sum()
)


proteins_enriched = int(
    protein_stats["protein_enriched"].sum()
)


print(
    "\nProteins with sufficient MSS measurements:",
    proteins_tested
)

print(
    "Proteins meeting enrichment criteria:",
    proteins_enriched
)

print(
    "Proteins lacking sufficient measurements:",
    int(
        protein_stats["wilcoxon_pvalue"]
        .isna()
        .sum()
    )
)


# ------------------------------------------------------------
# 8. LOAD TRANSCRIPTOMIC SCREENING EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TRANSCRIPTOMIC / PROTEOMIC INTEGRATION")
print("-" * 65)


if not TRANSCRIPTOMIC_PATH.exists():

    raise FileNotFoundError(
        "Patch 1.22 transcriptomic comparison "
        "table is missing."
    )


transcriptomic = pd.read_csv(
    TRANSCRIPTOMIC_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


required_columns = {
    "gene_id",
    "gene_name",
    "combined_transcriptomic_support"
}


if not required_columns.issubset(
    transcriptomic.columns
):

    raise RuntimeError(
        "Missing transcriptomic evidence columns."
    )


supported_mask = (

    transcriptomic[
        "combined_transcriptomic_support"
    ]

    .astype(str)
    .str.lower()
    .eq("true")

)


supported = transcriptomic[
    supported_mask
].copy()


if len(supported) != 948:

    raise RuntimeError(
        "Expected 948 transcriptomic screening genes. "
        f"Found {len(supported)}."
    )


supported["gene_name"] = (
    supported["gene_name"]
    .astype(str)
    .str.strip()
)


print("\nTranscriptomic screening genes:", len(supported))


# ------------------------------------------------------------
# 8A. Integrate by gene symbol
# ------------------------------------------------------------

integrated = supported.merge(

    protein_stats,

    left_on="gene_name",

    right_on="protein_gene_symbol",

    how="left",

    validate="many_to_one"

)


if len(integrated) != 948:

    raise RuntimeError(
        "Transcriptomic integration changed row count."
    )


print("\nGene-symbol integration: PASSED")


# ============================================================
# 9. CLASSIFY PROTEIN EVIDENCE
#
# CORRECTED:
# Convert all masks to explicit Boolean arrays
# to avoid nullable Boolean compatibility errors.
# ============================================================

print("\n" + "-" * 65)
print("PROTEIN EVIDENCE CLASSIFICATION")
print("-" * 65)


integrated["paired_assay_covered"] = (
    integrated["protein_gene_symbol"].notna()
)


enriched_mask = (

    integrated["protein_enriched"]

    .fillna(False)

    .astype(bool)

    .to_numpy(dtype=bool)

)


covered_mask = (

    integrated["paired_assay_covered"]

    .fillna(False)

    .astype(bool)

    .to_numpy(dtype=bool)

)


insufficient_mask = (

    covered_mask

    & integrated[
        "wilcoxon_pvalue"
    ].isna().to_numpy(dtype=bool)

)


integrated["protein_support_status"] = np.select(

    [

        enriched_mask,

        insufficient_mask,

        covered_mask

    ],

    [

        "MSS protein enrichment supported",

        "Measured; insufficient MSS observations",

        "Measured; enrichment criteria not met"

    ],

    default="Not represented in paired assay"

)


print("\nProtein evidence categories:")

print(
    integrated[
        "protein_support_status"
    ].value_counts().to_string()
)


# ------------------------------------------------------------
# 10. EXAMINE LEADING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING GENE PROTEIN EVIDENCE")
print("-" * 65)


leading_genes = [
    "CDH3",
    "CLDN1",
    "KRT80",
    "ETV4",
    "FOXQ1",
    "ESM1",
    "CEMIP",
    "INHBA",
    "SIM2",
    "WNT2"
]


leading = integrated[
    integrated["gene_name"].isin(
        leading_genes
    )
].copy()


gene_order = {
    gene: index
    for index, gene in enumerate(
        leading_genes
    )
}


leading["gene_order"] = (
    leading["gene_name"].map(
        gene_order
    )
)


leading = leading.sort_values(
    "gene_order"
)


display_columns = [
    "gene_name",
    "paired_assay_covered",
    "mss_measurements",
    "median_tumor_normal_log2ratio",
    "fraction_tumor_higher",
    "wilcoxon_fdr",
    "protein_support_status"
]


print("\n")

print(
    leading[display_columns]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)


print("\nIMPORTANT:")

print(
    "Missing paired-proteomics evidence "
    "does not establish protein absence."
)


# ------------------------------------------------------------
# 11. IDENTIFY PROTEOMICS-SUPPORTED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROTEOMICS-SUPPORTED SCREENING GENES")
print("-" * 65)


protein_supported = integrated[

    integrated["protein_support_status"]
    .eq("MSS protein enrichment supported")

].copy()


protein_supported = (
    protein_supported
    .sort_values(
        [
            "wilcoxon_fdr",
            "median_tumor_normal_log2ratio"
        ],
        ascending=[True, False]
    )
)


print(
    "\nTranscriptomic screening genes with "
    "MSS protein-enrichment support:",
    len(protein_supported)
)


if not protein_supported.empty:

    print("\nTop 20:")

    print(

        protein_supported[
            [
                "gene_name",
                "baseline_log2fc",
                "mss_measurements",
                "median_tumor_normal_log2ratio",
                "fraction_tumor_higher",
                "wilcoxon_fdr"
            ]
        ]

        .head(20)

        .to_string(
            index=False,
            float_format=lambda x: f"{x:.4g}"
        )

    )

else:

    print(
        "No proteins passed the prespecified "
        "combined screening criteria."
    )


# ------------------------------------------------------------
# 12. GENERATE PROTEOMICS FIGURE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING PROTEOMICS FIGURE")
print("-" * 65)


plot_data = protein_stats.dropna(

    subset=[
        "median_tumor_normal_log2ratio",
        "wilcoxon_fdr"
    ]

).copy()


plot_data["neg_log10_fdr"] = (

    -np.log10(

        plot_data["wilcoxon_fdr"]
        .clip(
            lower=np.finfo(float).tiny
        )

    )

)


fig, ax = plt.subplots(
    figsize=(10, 7)
)


ax.scatter(

    plot_data[
        "median_tumor_normal_log2ratio"
    ],

    plot_data[
        "neg_log10_fdr"
    ],

    s=12,

    alpha=0.4

)


ax.axvline(
    1,
    linestyle="--",
    linewidth=0.9
)

ax.axvline(
    0,
    linestyle=":",
    linewidth=0.8
)

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    linewidth=0.9
)


ax.set_xlabel(
    "Median reported tumor-normal protein log2 ratio"
)

ax.set_ylabel(
    "-log10 adjusted Wilcoxon p-value"
)

ax.set_title(
    "CPTAC Colon — MSS Paired Protein Enrichment"
)

ax.grid(alpha=0.2)

fig.tight_layout()


figure_path = (
    FIGURES /
    "cptac_mss_protein_enrichment_volcano.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

plt.close(fig)


print("\nProteomics figure saved.")


# ------------------------------------------------------------
# 13. SAVE ANALYSIS OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PROTEOMICS RESULTS")
print("-" * 65)


identifier_audit.to_csv(

    META / "cptac_colon_identifier_audit.csv",

    index=False

)


protein_stats.to_csv(

    RESULTS / "cptac_mss_paired_protein_statistics.csv",

    index=False

)


integrated.to_csv(

    RESULTS / "cptac_mss_transcriptomic_integration.csv",

    index=False

)


protein_supported.to_csv(

    RESULTS / "cptac_mss_protein_supported_candidates.csv",

    index=False

)


leading.to_csv(

    RESULTS / "cptac_mss_leading_gene_review.csv",

    index=False

)


print("\nAll proteomics analysis tables saved.")


# ------------------------------------------------------------
# 14. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING ANALYSIS PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "2.2",

    "patch_version": "2.2-corrected",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "study": "CPTAC Prospective Colon Cancer",

    "data_source": "LinkedOmics",

    "msi_group": "Published MSS",

    "mss_patients": int(
        len(mss_patients)
    ),

    "reported_measurement": (
        "Paired tumor-normal TMT protein log2 ratio"
    ),

    "minimum_observed_mss_patients":
        MIN_OBSERVED_MSS,

    "statistical_test": (
        "Two-sided Wilcoxon signed-rank test "
        "against zero"
    ),

    "additional_test": (
        "Two-sided binomial sign test"
    ),

    "multiple_testing": (
        "Benjamini-Hochberg FDR across "
        "all tested paired proteins"
    ),

    "protein_enrichment_criteria": {

        "wilcoxon_fdr": 0.05,

        "minimum_median_log2ratio": 1.0,

        "minimum_fraction_positive": 0.70

    },

    "proteins_tested": int(
        proteins_tested
    ),

    "proteins_enriched": int(
        proteins_enriched
    ),

    "transcriptomic_screening_genes": 948,

    "integrated_protein_supported_genes": int(
        len(protein_supported)
    ),

    "sample_identifier_mapping": (
        "CPTAC patient ID extracted from "
        "tumor, normal and paired sample labels"
    ),

    "clinical_identifier_fix": (
        "Standardized saved patient identifier "
        "column to cptac_patient_id"
    ),

    "protein_imputation_performed": False,

    "tcga_cptac_patient_level_merging": False,

    "separate_tumor_normal_subtraction": False,

    "limitations": [

        "Paired ratios are reported processed "
        "TMT measurements, not absolute "
        "protein concentrations.",

        "Minimum observation count is an "
        "exploratory screening criterion.",

        "Wilcoxon signed-rank testing assumes "
        "an appropriately symmetric distribution "
        "of paired differences for location inference.",

        "Missing proteomic measurements "
        "may not be random.",

        "Tumor-adjacent normal tissue is not "
        "equivalent to a healthy-organ "
        "safety atlas.",

        "Protein-level enrichment does not "
        "establish extracellular accessibility.",

        "Proteins absent from the paired-ratio "
        "matrix may still be measured in the "
        "separate tumor and normal matrices.",

        "Sample identifier correspondence "
        "does not establish identical "
        "quantification procedures.",

        "Gene-symbol integration is provisional "
        "and requires identifier validation.",

        "No therapeutic targets have been validated."

    ]

}


provenance_path = (
    META /
    "cptac_mss_protein_analysis_provenance.json"
)


with open(
    provenance_path,
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


print("\nAnalysis provenance saved.")


# ------------------------------------------------------------
# 15. CREATE PHASE 2 RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 2 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_Proteomics_Patch22_{timestamp}.zip"

)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)


print("\nCheckpoint created.")

print(
    f"Checkpoint size: {checkpoint_mib:.2f} MiB"
)

print("Checkpoint path:")

print(checkpoint)


# ------------------------------------------------------------
# 16. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 2.2 SUMMARY")
print("=" * 65)


print("\nPublished MSS patients:", len(mss_patients))

print(
    "Proteins in paired assay:",
    len(paired)
)

print(
    "Proteins eligible for statistical testing:",
    proteins_tested
)

print(
    "MSS-enriched proteins:",
    proteins_enriched
)


print("\nTranscriptomic screening genes:", 948)

print(
    "Genes with protein-enrichment support:",
    len(protein_supported)
)


print("\nSaved outputs:")

print("1. cptac_colon_identifier_audit.csv")

print("2. cptac_mss_paired_protein_statistics.csv")

print("3. cptac_mss_transcriptomic_integration.csv")

print("4. cptac_mss_protein_supported_candidates.csv")

print("5. cptac_mss_leading_gene_review.csv")

print("6. cptac_mss_protein_analysis_provenance.json")


print("\nFigure:")

print("cptac_mss_protein_enrichment_volcano.png")


print("\nResearch checkpoint: CREATED")

print(
    f"Checkpoint size: {checkpoint_mib:.2f} MiB"
)


print("\nClinical identifier fix: APPLIED")

print("No source measurements modified.")

print("No protein imputation performed.")

print("No TCGA/CPTAC patient-level merging.")


print("\nPATCH 2.2 COMPLETE")

print("=" * 65)